In [1]:
import app
import os
import string
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box
import rasterio as rio
import rasterio.plot as rplt
import rasterio.windows  as rw
from rasterio.transform import Affine
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
import xarray as xr

app.setup_logger(use_console_handler=True, use_file_handler=False)

In [2]:
base_directory = r"D:\PhD\21_Experiments\TidesDamageDriver"
drainages_folder = os.path.join(
    base_directory, "02_processed", "03_drainages"
)
new_folder = os.path.join(
    base_directory, "03_cleaned", "drainages"
)

years = [2016, 2018, 2019, 2020]

In [3]:
gdf_v = app.lakes.plotting.get_geodataframes(drainages_folder, ["_values."])[0]
gdf_v.sort_values(by="lon", inplace=True)
gdf_v.reset_index(drop=True, inplace=True)

In [4]:
gdf_v["label"] = list(string.ascii_uppercase)[: len(gdf_v)]
gdf_v.columns

Index(['index', 'criteria', 'window', 'lake id', 'type', 'tile', 'ifile_0',
       'area', 'file_0', 'date-0', 'sat-0', 'start-0', 'end-0', 'ifile_1',
       'file_1', 'date-1', 'sat-1', 'start-1', 'end-1', 'lon', 'lat', 'days',
       'days_long', 'area-0', 'status', 'reason', 'median-0', 'std-0',
       'fraction-0', 'fraction_d', 'mean-0', 'volume-0', 'median-1', 'mean-1',
       'volume-1', 'std_depth', 'year', 'justificat', 'start_date', 'end_date',
       'dmg', 'act', 'lai', 'fuerst', 'dmg_cat', 'act_cat', 'fuerst+lai',
       'geometry', 'label'],
      dtype='object')

In [5]:
gdf_clean = gdf_v[["label", "index", "year", "lon", "lat", "start_date", "end_date", 'mean-0', 'median-0', 'area-0', 'volume-0', 'mean-1', 'median-1', 
       'volume-1', "dmg", "dmg_cat", "act", "act_cat", "lai", "fuerst", "fuerst+lai", "geometry"]].copy()
gdf_clean.to_crs(
       "EPSG:4326", inplace=True
)
gdf_clean

,label,index,year,lon,lat,start_date,end_date,mean-0,median-0,area-0,...,median-1,volume-1,dmg,dmg_cat,act,act_cat,lai,fuerst,fuerst+lai,geometry
0,A,29,2019,95.688019,-66.671201,2019-12-24,2019-12-31,1.130955,1.104492,66600.0,...,0.000000,0.000000,0.235667,high,0.269375,medium,NaN,NaN,NaN,"POLYGON ((95.6809 -66.67233, 95.68083 -66.6720..."
1,B,28,2019,95.737264,-66.660983,2019-12-22,2019-12-31,1.283588,1.089844,55800.0,...,0.335693,1411.193848,0.333090,high,0.458099,medium,1.0,0.0,1.0,"POLYGON ((95.73322 -66.6612, 95.73302 -66.6604..."
2,C,39,2019,95.759349,-66.646830,2020-01-07,2020-01-13,0.569657,NaN,NaN,...,NaN,6092.188454,0.333090,high,0.458099,medium,1.0,0.0,1.0,"POLYGON ((95.75933 -66.64674, 95.75935 -66.646..."
3,D,8,2016,96.040652,-66.571884,2017-01-27,2017-02-03,1.441006,1.357422,90900.0,...,0.222290,600.183105,0.356259,high,0.574090,medium,1.0,0.0,1.0,"POLYGON ((96.03223 -66.57217, 96.03216 -66.571..."
4,E,32,2019,97.782530,-66.594982,2020-01-30,2020-02-03,2.299795,2.486328,154800.0,...,0.626221,14769.579649,0.139619,medium,0.533078,medium,1.0,0.0,1.0,"POLYGON ((97.7797 -66.5975, 97.77961 -66.59723..."
5,F,40,2019,98.104002,-66.492315,2020-01-30,2020-02-03,1.125474,NaN,NaN,...,NaN,5646.196747,0.169961,high,0.513592,medium,1.0,1.0,2.0,"POLYGON ((98.10397 -66.49223, 98.10399 -66.492..."
6,G,41,2019,98.493578,-65.782618,2020-01-04,2020-01-11,0.583205,NaN,NaN,...,NaN,0.000000,0.430535,high,0.801361,high,1.0,0.0,1.0,"POLYGON ((98.49355 -65.78253, 98.49357 -65.782..."
7,H,17,2018,98.847890,-66.359584,2019-01-31,2019-02-08,0.914807,0.891113,99900.0,...,0.585693,7497.070205,0.284860,high,0.313371,medium,0.0,0.0,0.0,"POLYGON ((98.84161 -66.36032, 98.84131 -66.359..."
8,I,26,2019,98.870325,-66.357777,2020-01-20,2020-01-27,0.860971,0.859863,93600.0,...,0.876709,7370.727539,0.256579,high,0.386947,medium,0.0,0.0,0.0,"POLYGON ((98.86448 -66.35728, 98.86428 -66.356..."
9,J,23,2019,99.719303,-66.264245,2020-01-04,2020-01-11,0.992884,1.030273,59400.0,...,0.489014,2612.988281,0.135486,medium,0.681241,high,0.0,0.0,0.0,"POLYGON ((99.71542 -66.26532, 99.71531 -66.265..."


In [6]:
gdf_clean.loc[gdf_clean["label"] == "R", "year"] = 2019
gdf_clean

,label,index,year,lon,lat,start_date,end_date,mean-0,median-0,area-0,...,median-1,volume-1,dmg,dmg_cat,act,act_cat,lai,fuerst,fuerst+lai,geometry
0,A,29,2019,95.688019,-66.671201,2019-12-24,2019-12-31,1.130955,1.104492,66600.0,...,0.000000,0.000000,0.235667,high,0.269375,medium,NaN,NaN,NaN,"POLYGON ((95.6809 -66.67233, 95.68083 -66.6720..."
1,B,28,2019,95.737264,-66.660983,2019-12-22,2019-12-31,1.283588,1.089844,55800.0,...,0.335693,1411.193848,0.333090,high,0.458099,medium,1.0,0.0,1.0,"POLYGON ((95.73322 -66.6612, 95.73302 -66.6604..."
2,C,39,2019,95.759349,-66.646830,2020-01-07,2020-01-13,0.569657,NaN,NaN,...,NaN,6092.188454,0.333090,high,0.458099,medium,1.0,0.0,1.0,"POLYGON ((95.75933 -66.64674, 95.75935 -66.646..."
3,D,8,2016,96.040652,-66.571884,2017-01-27,2017-02-03,1.441006,1.357422,90900.0,...,0.222290,600.183105,0.356259,high,0.574090,medium,1.0,0.0,1.0,"POLYGON ((96.03223 -66.57217, 96.03216 -66.571..."
4,E,32,2019,97.782530,-66.594982,2020-01-30,2020-02-03,2.299795,2.486328,154800.0,...,0.626221,14769.579649,0.139619,medium,0.533078,medium,1.0,0.0,1.0,"POLYGON ((97.7797 -66.5975, 97.77961 -66.59723..."
5,F,40,2019,98.104002,-66.492315,2020-01-30,2020-02-03,1.125474,NaN,NaN,...,NaN,5646.196747,0.169961,high,0.513592,medium,1.0,1.0,2.0,"POLYGON ((98.10397 -66.49223, 98.10399 -66.492..."
6,G,41,2019,98.493578,-65.782618,2020-01-04,2020-01-11,0.583205,NaN,NaN,...,NaN,0.000000,0.430535,high,0.801361,high,1.0,0.0,1.0,"POLYGON ((98.49355 -65.78253, 98.49357 -65.782..."
7,H,17,2018,98.847890,-66.359584,2019-01-31,2019-02-08,0.914807,0.891113,99900.0,...,0.585693,7497.070205,0.284860,high,0.313371,medium,0.0,0.0,0.0,"POLYGON ((98.84161 -66.36032, 98.84131 -66.359..."
8,I,26,2019,98.870325,-66.357777,2020-01-20,2020-01-27,0.860971,0.859863,93600.0,...,0.876709,7370.727539,0.256579,high,0.386947,medium,0.0,0.0,0.0,"POLYGON ((98.86448 -66.35728, 98.86428 -66.356..."
9,J,23,2019,99.719303,-66.264245,2020-01-04,2020-01-11,0.992884,1.030273,59400.0,...,0.489014,2612.988281,0.135486,medium,0.681241,high,0.0,0.0,0.0,"POLYGON ((99.71542 -66.26532, 99.71531 -66.265..."


In [7]:
gdf_clean.to_file(
    os.path.join(new_folder, "drainages.shp"
)
    , driver="ESRI Shapefile"
)
gdf_clean.to_csv(
    os.path.join(new_folder, "drainages.csv"),
    index=False,
)

2025-07-22 00:45:32,762 INFO     pyogrio._io Created 25 records


In [8]:
gdf_clean_c = gdf_clean.copy()
gdf_clean_c["geometry"] = gdf_clean_c.centroid
gdf_clean_c.to_file(
    os.path.join(new_folder, "drainages_centroids.shp"),
    driver="ESRI Shapefile",
)

C:\Users\jsommer1\AppData\Local\Temp\ipykernel_13976\2622755438.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_clean_c["geometry"] = gdf_clean_c.centroid
2025-07-22 00:45:33,054 INFO     pyogrio._io Created 25 records


In [13]:
gdf_clean_c["dmg_cat"].value_counts()

dmg_cat
high      17
medium     8
Name: count, dtype: int64

In [14]:
gdf_clean_c["act_cat"].value_counts()

act_cat
medium    20
high       4
low        1
Name: count, dtype: int64